# ClinHallu revision: Colab worker (identical on all 5 accounts)
Runtime -> GPU (T4). Change `ACCOUNT` per account (A1..A5). Each run trains on a private local copy and mirrors to Drive; claims are in `jobs/claims/`.

In [10]:
file_path = r"D:\Thesis code\ClinHallu\data\raw\c5\train.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    count = sum(1 for line in f if line.strip())

print(f"Total data points: {count}")

Total data points: 14000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/ClinHallu'   # shared folder shortcut, same path on every account
ACCOUNT = 'A1'                                     # <-- change per account

In [ ]:
import os
# Colab has no .git: record the frozen commit from BASELINE.md for provenance.
os.environ['GIT_COMMIT'] = 'PASTE_COMMIT_HASH'
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
!pip -q install -r {DRIVE_ROOT}/requirements/colab.txt
# OpenRouter key (only needed for judge/generation jobs) comes from Colab Secrets:
from google.colab import userdata
try: os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
except Exception: pass

In [ ]:
# one-time per Drive folder (any account): build revision data + job lists
!cd {DRIVE_ROOT} && python scripts/make_manifests.py > /dev/null && python scripts/make_jobs.py

In [ ]:
# dry run: see what this account would do
!python {DRIVE_ROOT}/scripts/colab_worker.py --drive-root {DRIVE_ROOT} --account {ACCOUNT} --dry-run

In [ ]:
# real run: stops starting jobs when there is not enough time left to finish one cleanly
!python {DRIVE_ROOT}/scripts/colab_worker.py --drive-root {DRIVE_ROOT} --account {ACCOUNT} \
    --max-hours 10 --poll-minutes 15

If the session dies, just re-run the last cell: the claim goes stale after 3 h and the run resumes from the last checkpoint mirrored to Drive.
Daily check: count `artifacts/adaptation/*/seed_*/DONE` against `jobs/jobs.json` (33 train + 27 eval).